# 2 - Process and Merge Datasets
Next, we want to process the ADCP and Science Datasets so that we can merge them into a single unified dataset, derive the individual profile dives, and run the vendor-suggested Quality Control algorithms.

In [ ]:
import os

import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
# For this example we will be using CP05MOAS-GL379 D00008, which was deployed at the Pioneer NES array from 2019-09-27 to 2019-11-19.
refdes = 'CP05MOAS-GL379-01-ADCPAM000'
eng_refdes = 'CP05MOAS-GL379-00-ENG000000'
deployment = 8

In [ ]:
dep_str = f"{deployment:04d}"  # always 4 digits, e.g., '0008'
glider_name = '-'.join(refdes.split('-')[:2]) # shorten ref des to get glider name

In [ ]:
# Load the ADCP Data
adcp = xr.open_dataset(f"../data/raw/{refdes}.deployment{dep_str}.adcp_data.nc").load()

# Load the Science Data
gdac = xr.open_dataset(f"../data/raw/{glider_name}.deployment{dep_str}.science.nc").load()

#### Merge the Datasets
Merging the glider ADCP and Science datasets is a multistep process. The Science dataset contains all of the different science sensors, along with the glider's built-in sensors and navigation, on a common timestamp. However, the different science sensors all sample at different rates. Thus, the science parameters in the dataset are sparse and filled with NaNs. In order to merge the Science and ADCP data streams, we need to do the following:
1. Split the separate science and glider parameters into different datasets based on their source sensor
2. Drop the NaNs from each science parameter, so each one is interpolated from only its own measurements
3. Linearly interpolate each science parameter to the ADCP time. ADCP times inside a gap in a parameter's measurements longer than `max_gap` (60 seconds by default), e.g. while the sensor is off, are left as NaN rather than interpolated across
4. Merge the interpolated, grouped parameter sets to the ADCP dataset

In [ ]:
from ooicgsn_glider_dvl.merge import add_waypoints, downcast, merge_datasets, netcdf_encoding

In [ ]:
merged = merge_datasets(adcp, gdac)

# Store the (time, bin) ADCP variables as float32 to halve the memory use
merged = downcast(merged)
merged

#### Identify Individual Profiles/Dives
The ADCP on OOI gliders is only active during dives. The algorithm to separate the profiles looks for where the absolute difference between two depths exceeds 2 meters and the sampling interval is greater than 4 seconds (the ADCP samples at 2 seconds). Both thresholds can be changed with the `min_depth_change` and `min_time_gap` arguments of `add_profiles`.

In [ ]:
# Next, identify the individual profiles
from ooicgsn_glider_dvl.profiles import add_profiles

In [ ]:
merged = add_profiles(merged)
merged

### Vendor QA/QC 

First, utilize the TRDI Data QA-QC Model rev12-1 thresholds to derive a first-pass QC flag. We assume that the Explorer DVL 600kHz is comparable to the WH600kHz (Wide-Band) for setting the thresholds.
I think the config settings are: 2.0 meter depth cell size, 0.1 ping interval time, 10 pings per ensemble

| Parameter | Good | Suspect | Bad |
| --------- | ---- | ------- | --- |
| Error Velocity Threshold | <=6.3 cm/s | >6.3 cm/s, <=12.6 cm/s | >=12.6 cm/s |
| Correlation Magnitude | >=115 | <115, >=63 | <63 |
| Percent Good | >=63% | <63%, >=50% | <50% |
| BIT Errors | 0 | >=1 | N/A |

In [ ]:
from ooicgsn_glider_dvl import qc

In [ ]:
# Run the individual tests (make sure inputs and units match)
percent_good = qc.percent_good_qc(merged, 63, 50)
corr_mag = qc.correlation_magnitude_qc(merged, 115, 63)
error_vel = qc.error_velocity_qc(merged, 6.3/100, 12.6/100)

# Merge the test results
qc_flag = qc.merge_qc([percent_good, corr_mag, error_vel])

In [ ]:
qc_flag

In [ ]:
# Now add the qc_flags to the dataset
merged['vendor_qc_flag'] = (['time','bin'], qc_flag)
merged['vendor_qc_flag'].attrs = {
            'long_name': 'TRDI QC Summary Flag',
            'standard_name': 'aggregate_quality_flag',
            'comment': ('Summary of the TRDI QC tests as a QARTOD style summary flag, where '
                        'the values are 1 == pass, 2 == not evaluated, 3 == suspect or of high interest, '
                        '4 == fail, and 9 == missing.'),
            'flag_values': np.array([1, 2, 3, 4, 9], dtype=np.int8),
            'flag_meanings': 'pass not_evaluated suspect_or_of_high_interest fail missing'
        }

#### Add Waypoints
The next step is to add the waypoint data into the merged dataset. The waypoint data is included in the glider engineering data and is not in either the science or adcp datasets. The waypoint data is necessary to calculate the net water-column velocity, which is achieved by comparing the expected waypoint with the actual surfacing location. Each waypoint applies from the time it is set until the next waypoint is set. A waypoint of (0, 0) is the glider's placeholder before a waypoint is set (e.g. after a reboot), so ADCP times while it applies have no waypoint (NaN).

In [ ]:
# Use the recovered engineering data, or the telemetered data if the recovered data wasn't available
eng_file = f"../data/raw/deployment{dep_str}_{eng_refdes}-recovered_host-glider_eng_recovered.csv"
if not os.path.exists(eng_file):
    eng_file = f"../data/raw/deployment{dep_str}_{eng_refdes}-telemetered-glider_eng_telemetered.csv"
    print(f"No recovered engineering data, using the telemetered data: {eng_file}")

# Only the waypoint, dead-reckoning, and GPS columns are needed from the engineering data
from ooicgsn_glider_dvl.currents import CURRENT_VARIABLES

glider = pd.read_csv(
    eng_file,
    usecols=['c_wpt_lat', 'c_wpt_lon'] + CURRENT_VARIABLES,
    parse_dates=['time'],
)
glider.head()

In [ ]:
merged = add_waypoints(merged, glider)

#### Add the Depth-Averaged Current
Between surfacings, the glider makes many dives and dead reckons its position from its heading, its speed through the water, and the current it estimated at the previous surfacing. When it surfaces, the displacement between its first GPS fix and its dead-reckoned position is the drift that the dead reckoning did not account for. Dividing the displacement by the time spent underwater gives the correction to the current estimate, and adding it to the current used in the dead reckoning gives the depth-averaged current over all of the dives since the last surfacing.

The positions are the glider's own dead-reckoned surfacing position (`m_dr_surf_x/y_lmc`) and first GPS fix (`m_gps_fix_x/y_lmc`), in meters east and north. The glider's `m_lat` and `m_lon` can't be used for this, because after surfacing they ramp from the dead-reckoned position to the GPS fix.

Each ADCP time between the start of the dives and the surfacing is given the depth-averaged current calculated at that surfacing.

In [ ]:
from ooicgsn_glider_dvl.currents import add_depth_averaged_current, surfacing_currents

# Calculate the depth-averaged current at each surfacing
surfacings = surfacing_currents(glider)
surfacings.head()

In [ ]:
merged = add_depth_averaged_current(merged, surfacings)

In [ ]:
# Save the results, compressing each variable
os.makedirs("../data/processed", exist_ok=True)
merged.to_netcdf(
    f"../data/processed/{glider_name}.deployment{dep_str}.merged.nc",
    format='netcdf4',
    engine='h5netcdf',
    encoding=netcdf_encoding(merged),
)

# Save the depth-averaged current at each surfacing
surfacings.to_csv(f"../data/processed/{glider_name}.deployment{dep_str}.depth_averaged_current.csv", index=False)